In [ ]:
!source myenv/bin/activate

In [ ]:
%pip install tokenizers matplotlib seaborn

In [ ]:
%pip uninstall -y torch torchvision torchaudio
%pip install torch torchvision --index-url https://download.pytorch.org/whl/cu124

In [ ]:
!pip install -q opencv-python matplotlib numpy pillow accelerate bitsandbytes

In [ ]:
!pip install -q accelerate

In [ ]:
%pip uninstall -y transformers
!python -m pip install --upgrade --force-reinstall git+https://github.com/huggingface/transformers

In [ ]:
import torch
import cv2
import numpy as np
import math
from PIL import Image, ImageDraw, ImageFont
import re
from collections import defaultdict
import matplotlib.pyplot as plt
import json
import os
import requests
from io import BytesIO
from transformers import (
    AutoProcessor, 
    AutoModelForCausalLM, 
    Qwen2_5_VLForConditionalGeneration,
    AutoModelForVision2Seq,
    Sam3Model, 
    Sam3Processor,
    Qwen2VLForConditionalGeneration
)

In [ ]:
class GeoCalculator:
    """
    Handles geometric calculations and pixel-to-meter conversions 
    based on the provided spatial resolution (GSD).
    """
    def __init__(self, spatial_resolution_m=1.0):
        self.gsd = spatial_resolution_m

    def pixel_to_meter(self, pixel_dist):
        return pixel_dist * self.gsd

    def pixel_area_to_meter_sq(self, pixel_area):
        # Area scales with the square of the resolution (m/px)^2
        return pixel_area * (self.gsd ** 2)

    def calculate_polygon_area(self, obb_or_mask):
        """
        Computes real-world area (m^2) from an OBB or Mask.
        """
        if isinstance(obb_or_mask, tuple): # It's an OBB ((cx, cy), (w, h), ang)
            (cx, cy), (w, h), angle = obb_or_mask
            pixel_area = w * h
        else: # It's a mask
            pixel_area = np.sum(obb_or_mask > 0)
            
        return self.pixel_area_to_meter_sq(pixel_area)

    def get_obb_from_mask(self, mask_bool):
        """
        Converts a binary mask (H, W) into an Oriented Bounding Box (OBB).
        Returns: ((cx, cy), (w, h), angle) or None if noise.
        """
        mask_uint8 = mask_bool.astype(np.uint8) * 255
        contours, _ = cv2.findContours(mask_uint8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        
        if not contours: 
            return None
        
        # Select largest contour to ignore small segmentation noise
        largest_cnt = max(contours, key=cv2.contourArea)
        
        # Filter extremely small artifacts (e.g., < 10 pixels area)
        if cv2.contourArea(largest_cnt) < 10:
            return None
            
        # cv2.minAreaRect returns ((center_x, center_y), (width, height), angle)
        rect = cv2.minAreaRect(largest_cnt)
        return rect


class RS_Pipeline:
    def __init__(self, device="cuda" if torch.cuda.is_available() else "cpu"):
        self.device = device
        print(f"--- Initializing RS Pipeline on {self.device} ---")
        
        # A. Load VLM 
        print("Loading Qwen VLM...")
        self.vlm_model_id = "Qwen/Qwen2.5-VL-3B-Instruct"
        try:
            self.vlm_processor = AutoProcessor.from_pretrained(self.vlm_model_id, trust_remote_code=True)
            self.vlm_model = AutoModelForVision2Seq.from_pretrained(
                 self.vlm_model_id,
                device_map="auto",
                torch_dtype=torch.bfloat16, 
               trust_remote_code=True
            ).eval()
        except Exception as e:
            print(f"Error loading Qwen: {e}. Ensure transformers is updated.")
            raise e

        # B. Load SAM 3
        # Reusing the loading logic from vlm_sam3 (1).ipynb
        print("Loading SAM 3...")
        self.sam_model_id = "facebook/sam3"
        try:
            self.sam_processor = Sam3Processor.from_pretrained(self.sam_model_id)
            self.sam_model = Sam3Model.from_pretrained(self.sam_model_id).to(self.device).eval()
        except Exception as e:
            print(f"Error loading SAM 3: {e}. Check if 'facebook/sam3' is accessible.")
            raise e
            
        print("Pipeline Initialization Complete.")

    def ask_qwen(self, image, prompt, system_prompt=None, max_tokens=512):
        messages = []
        
        if system_prompt:
            messages.append({"role": "system", "content": system_prompt})
            
        messages.append({
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": prompt},
            ]
        })
        
        text_prompt = self.vlm_processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        
        inputs = self.vlm_processor(
            text=[text_prompt], 
            images=[image], 
            return_tensors="pt", 
            padding=True
        ).to(self.device)
        
        output_ids = self.vlm_model.generate(**inputs, max_new_tokens=max_tokens)
        
        generated_ids = [
            output_ids[len(input_ids):] 
            for input_ids, output_ids in zip(inputs.input_ids, output_ids)
        ]
        return self.vlm_processor.batch_decode(
            generated_ids, skip_special_tokens=True, clean_up_tokenization_spaces=True
        )[0]

    def annotate_image_with_boxes(self, image, obbs):
        annotated_img = image.copy()
        draw = ImageDraw.Draw(annotated_img)

        # Load font (fallback to default if needed)
        try:
            font = ImageFont.truetype("arial.ttf", size=20)
        except:
            font = ImageFont.load_default()

        id_map = {}
        
        for idx, obb in enumerate(obbs, start=1):
            if obb is None: continue

            # 1. Get Box Points from OBB ((cx,cy), (w,h), angle)
            box_points = cv2.boxPoints(obb)
            box_points = np.int32(box_points)
            
            # 2. Draw Polygon (Red, width=3)
            polygon = [tuple(pt) for pt in box_points]
            draw.polygon(polygon, outline="red", width=3)

            # 3. Draw ID Label with background
            center_x, center_y = obb[0]
            text = str(idx)
            
            # Calculate text background box
            bbox = draw.textbbox((center_x, center_y), text, font=font)
            draw.rectangle(
                (bbox[0]-2, bbox[1]-2, bbox[2]+2, bbox[3]+2), 
                fill="white", 
                outline="red"
            )
            draw.text((center_x, center_y), text, fill="black", font=font, anchor="mm")
            
            id_map[idx] = obb
            
        return annotated_img, id_map


In [ ]:
class RS_Pipeline(RS_Pipeline):
    def expand_box(self, bbox, img_width, img_height, factor=0.2):
        """
        Expands the bounding box by a given factor, ensuring it stays within image bounds.
        """
        x1, y1, x2, y2 = bbox
        w, h = x2 - x1, y2 - y1
        
        dx, dy = w * factor, h * factor
        
        new_x1 = max(0, x1 - dx)
        new_y1 = max(0, y1 - dy)
        new_x2 = min(img_width, x2 + dx)
        new_y2 = min(img_height, y2 + dy)
        
        return [int(new_x1), int(new_y1), int(new_x2), int(new_y2)]

    def bbox_to_obb(self, bbox):
        """Converts [x1, y1, x2, y2] to OBB format [cx, cy, w, h, angle]."""
        x1, y1, x2, y2 = bbox
        w, h = x2 - x1, y2 - y1
        cx, cy = x1 + w / 2, y1 + h / 2
        return ((cx, cy), (w, h), 0.0)


    def refine_with_sam3_crop(self, image, hbb, target_label):

        img_w, img_h = image.size
        
        # 1. Expand box for context
        crop_box = self.expand_box(hbb, img_w, img_h, factor=0.2)
        cx1, cy1, cx2, cy2 = crop_box
        cropped_img = image.crop((cx1, cy1, cx2, cy2))

        inputs = self.sam_processor(
            images=cropped_img, 
            text=[target_label], # Pass the specific class label here
            return_tensors="pt"
        ).to(self.device)
        
        with torch.no_grad():
            outputs = self.sam_model(**inputs)
        
        results = self.sam_processor.post_process_instance_segmentation(
            outputs, 
            threshold=0.4, # Confidence threshold
            target_sizes=[(cropped_img.height, cropped_img.width)]
        )[0]
        
        masks = results["masks"]
        scores = results["scores"]
        
        if len(masks) == 0:
            print(f"[SAM3] No masks found for '{target_label}' - using Qwen HBB as fallback.")
            qx1, qy1, qx2, qy2 = hbb
            qw, qh = qx2 - qx1, qy2 - qy1
            qcx, qcy = qx1 + qw/2, qy1 + qh/2
            fallback_obb = ((qcx, qcy), (qw, qh), 0.0) 
            return {
                "obb": fallback_obb,
                "score": 0.99, # Assign high confidence to ensure it passes threshold
                "is_fallback": True
            }
            

        # Pick the best mask (highest score)
        best_idx = torch.argmax(scores).item()
        mask_bool = masks[best_idx].cpu().numpy()
        confidence = scores[best_idx].item()
        
        # 5. Convert Mask to OBB in CROP coordinates
        # Use GeoCalculator helper if available, else manual
        try:
            calc = GeoCalculator(spatial_resolution_m=1.0) # Dummy GSD for extraction
            obb_crop = calc.get_obb_from_mask(mask_bool)
        except:
             # Fallback if GeoCalculator isn't perfectly aligned
             # (cx, cy), (w, h), angle
             contours, _ = cv2.findContours(mask_bool.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
             if not contours: return None
             largest_cnt = max(contours, key=cv2.contourArea)
             obb_crop = cv2.minAreaRect(largest_cnt)

        if not obb_crop:
            return None

        # 6. Transform OBB back to ORIGINAL image coordinates
        (crop_center_x, crop_center_y), (obb_w, obb_h), angle = obb_crop
        
        orig_center_x = crop_center_x + cx1
        orig_center_y = crop_center_y + cy1
        
        obb_orig = ((orig_center_x, orig_center_y), (obb_w, obb_h), angle)
        
        return {
            "obb": obb_orig,
            "score": confidence,
            "hbb": hbb # Original coarse box
        }

    def ensure_detections_for_query(self, image, query, current_detections, gsd=1.0):

        # 1. Find what the question is looking for
        needed_targets = self.extract_target_classes(image, query)
        
        # 2. Check what we already have
        existing_labels = {d['label'] for d in current_detections}
        
        # 3. Identify missing targets
        missing_targets = [t for t in needed_targets if t not in existing_labels]
        
        if not missing_targets:
            return current_detections # We have everything we need
            
        print(f"   [Dynamic Grounding] VQA needs {missing_targets}. Scanning...")
        new_detections = self.process_grounding(
            image, 
            query=query, 
            gsd=gsd,
            forced_targets=missing_targets
        )
    
        updated_detections = current_detections + new_detections
        return updated_detections
        
    def extract_target_classes(self, image, query):
        """
        Updated extraction logic: Identifies MULTIPLE target categories 
        and returns them as a clean list.
        """
        prompt = (
            f"Analyze the user query: '{query}'.\n"
            "Identify ALL distinct object categories that need to be detected.\n"
            "Return a comma-separated list of category names (singular form). Do not add punctuation or filler text.\n"
            "Example 1: 'Find the red cars and the pool' -> car, swimming pool\n"
            "Example 2: 'Locate the planes' -> plane\n"
            "Target Categories:"
        )
        
        response = self.ask_qwen(image, prompt, max_tokens=32)
        clean_resp = re.sub(r"[\[\]\']", "", response)
        classes = [c.strip().lower() for c in clean_resp.split(',') if c.strip()]
        
        print(f"   [Extraction] Query: '{query}' -> Targets: {classes}")
        return classes

    def detect_hbb_with_qwen(self, image, description, norm_scale=1000):
        """
        Uses Qwen to detect objects based on the FULL description/query.
        Returns a list of [x1, y1, x2, y2] boxes.
        """
        # Construct prompt for Qwen (similar to qwen_bb approach)
        prompt_text = (
            f"Detect the object described by: '{description}'.\n"
            f"Return the bounding boxes in [x1, y1, x2, y2] format.\n"
            f"Use a coordinate scale of 0-{norm_scale}.\n"
            "Return ONLY the Python list of lists. Example: [[100, 200, 500, 600]]"
        )
        
        # Ask Qwen
        response = self.ask_qwen(image, prompt_text, max_tokens=128)
        
        # Parse response
        try:
            # Look for list of lists [[...]]
            match = re.search(r"\[\[.*?\]\]", response)
            if match:
                hbb_list_norm = eval(match.group(0))
            else:
                # Fallback for single list [...]
                match_single = re.search(r"\[\d+.*?\]", response)
                if match_single:
                    hbb_list_norm = [eval(match_single.group(0))]
                else:
                    return []
        except:
            print(f"   [Warning] Parsing failed for Qwen detection.")
            return []

        # Convert Normalized -> Absolute Pixels
        w, h = image.size
        abs_hbbs = []
        for box in hbb_list_norm:
            if len(box) == 4:
                x1 = box[0] * w / norm_scale
                y1 = box[1] * h / norm_scale
                x2 = box[2] * w / norm_scale
                y2 = box[3] * h / norm_scale
                abs_hbbs.append([x1, y1, x2, y2])
                
        return abs_hbbs
    
    def process_captioning(self, image, instruction):
        """
        Task 1: Captioning with 'Generate -> Compress' strategy for high BLEU.
        """
        print(f"--- Task: Captioning ---")
        
        draft_prompt = (
            f"{instruction}\n"
            "Draft a comprehensive description listing all visible objects, "
            "their counts, colors, and relative positions. Be verbose."
        )
        long_caption = self.ask_qwen(image, draft_prompt, max_tokens=400)
        
        # Step 2: High-Precision Compression (Targeting ~60 words)
        # We ask the model to summarize its own draft
        compress_prompt = (
            f"Here is a detailed description of the image:\n'{long_caption}'\n\n"
            f"User Instruction: {instruction}\n"
            "Task: Summarize this description into a single, high-density caption.\n"
            "Constraints:\n"
            "1. Target length: Approximately 60 words.\n"
            "2. Do NOT exceed 80 words.\n"
            "3. You must use complete, grammatical sentences.\n"
            "4. Retain ALL specific details and relative positions.\n"
            "5. Remove 'fluff'"
        )
        
        final_caption = self.ask_qwen(image, compress_prompt, max_tokens=128)
        
        print(f"-> Draft Length: {len(long_caption.split())} words")
        print(f"-> Final Length: {len(final_caption.split())} words")
        
        return final_caption


    
    def process_grounding(self, image, query, gsd=1.0, score_threshold=0.4, forced_targets=None):

        if forced_targets:
            target_classes = forced_targets
            description = query if query else f"Find {', '.join(forced_targets)}"
            print(f"--- Task: Grounding (Dynamic for: {target_classes}) ---")
        else:
            print(f"--- Task: Grounding (Query: '{query}') ---")
            target_classes = self.extract_target_classes(image, query)
            description = query

        all_detections = []
        global_id = 1
        calc = GeoCalculator(spatial_resolution_m=gsd)
        
        # 2. Coarse Detection (Qwen sees full context)
        print(f"   -> Step 1: Qwen detecting HBBs using description: '{description}'...")
        coarse_hbbs = self.detect_hbb_with_qwen(image, description)
        
        if not coarse_hbbs:
            print("      No coarse HBBs found by Qwen.")
            return []
            
        print(f"      Found {len(coarse_hbbs)} coarse box(es). Refining...")

        # 3. Refinement Loop (SAM sees specific target + cropped context)
        # We iterate through targets AND boxes because Qwen might return mixed boxes
        for target in target_classes:
            print(f"   -> Step 2: Refining for target class '{target}'...")
            
            for hbb in coarse_hbbs:
                result = self.refine_with_sam3_crop(image, hbb, target)
                
                # Check score (Fallback gets 0.99, so it passes)
                if result and result['score'] > score_threshold:
                    
                    # De-duplication
                    is_duplicate = False
                    new_center = result['obb'][0]
                    for existing in all_detections:
                        ex_center = existing['center_point']
                        dist = ((ex_center[0] - new_center[0])**2 + (ex_center[1] - new_center[1])**2)**0.5
                        if dist < 20: 
                            is_duplicate = True
                            break
                    
                    if not is_duplicate:
                        result['id'] = global_id
                        result['label'] = target
                        result['area_m2'] = calc.calculate_polygon_area(result['obb'])
                        result['center_point'] = result['obb'][0]
                        all_detections.append(result)
                        global_id += 1

        print(f"   -> Found {len(all_detections)} verified objects.")
        
        if all_detections:
            final_obbs = [d['obb'] for d in all_detections]
            annotated_preview, _ = self.annotate_image_with_boxes(image, final_obbs)
            
            plt.figure(figsize=(12, 12))
            plt.imshow(annotated_preview)
            plt.axis('off')
            plt.title(f"Grounding Result: {description}")
            plt.show()
            
        return all_detections

In [ ]:
class RS_Pipeline(RS_Pipeline):
    def format_grounding_context(self, detections):
        """
        Converts the list of detection dicts into a text context for the VLM.
        """
        if not detections:
            return "No objects were detected in the grounding phase."

        grouped = defaultdict(list)
        for det in detections:
            grouped[det['label']].append(det)
            
        lines = ["Grounding Phase Results:"]

        summary_parts = []
        for label, dets in grouped.items():
            summary_parts.append(f"{len(dets)} {label}(s)")
        lines.append("Summary: Found " + ", ".join(summary_parts) + ".")
        lines.append("-" * 30)

        for label, dets in grouped.items():
            lines.append(f"Category: '{label}'")
            for det in dets:
                cx, cy = det['center_point']
                lines.append(
                    f"  - ID {det['id']}: "
                    f"Area={det['area_m2']:.2f}m2, "
                    f"Center=({int(cx)}, {int(cy)}), "
                    f"Angle={det['obb'][2]:.1f}"
                )
        return "\n".join(lines)


# this needs to be made more robust/ we need to prompt engineer the hell out of qwen to turn the original question to a version which fits the requirments of this function 
    
    def analyze_query_intent(self, query):
        """
        Heuristic classifier to decide if we can skip the VLM.
        Motivation: to offload as much of the work to the tool so that VLM won't hallucinate 
        """
        q = query.lower().strip()
        
        if re.search(r"^(how many|count|number of)", q):
            return "COUNT"
            
        if re.search(r"^(is there|are there|do you see|can you find)", q):
            return "EXISTENCE"
            
        if re.search(r"(largest|smallest|biggest|closest|farthest)", q):
            return "EXTREMA"
            
        return "COMPLEX"

    def solve_heuristic_vqa(self, query, detections):
        """
        Attempts to answer the question programmatically.
        Returns: (Answer String, Success Bool)
        """
        intent = self.analyze_query_intent(query)
        query_lower = query.lower()
        
        # Filter detections based on words in the query
        # e.g., if query is "How many cars?", we count items where label='car'
        relevant_dets = [d for d in detections if d['label'].lower() in query_lower]
        
        # Fallback: If we can't match the label in the text, use ALL detections 
        # (assuming the grounding step was specific to this query).

        if intent == "COUNT":
            return f"{len(relevant_dets)}", True
        elif intent == "EXISTENCE":
            return ("Yes" if len(relevant_dets) > 0 else "No"), True
        elif intent == "EXTREMA":
            if not relevant_dets: return "None found", True
            if "largest" in query_lower or "biggest" in query_lower:
                target = max(relevant_dets, key=lambda x: x['area_m2'])
                return f"Object ID {target['id']} ({target['label']}) Area={target['area_m2']:.2f}m2", True
            if "smallest" in query_lower:
                target = min(relevant_dets, key=lambda x: x['area_m2'])
                return f"Object ID {target['id']} ({target['label']}) Area={target['area_m2']:.2f}m2", True

        return None, False 

# Updated for Pure VLM pipeline
    def solve_numeric_vqa_run(self, image, query, detections, gsd=1.0):
        print(f"--- Task: Numeric VQA ('{query}') ---")
        
        detections = self.ensure_detections_for_query(image, query, detections, gsd)
        
        # 2. Prepare Context
        context_str = self.format_grounding_context(detections)
        
        # 3. Define Strict System Behavior
        sys_prompt = (
            "You are a helpful AI assistant acting as a calculator. "
            "You will be provided with a list of detected objects and their metadata (Area, Coordinates). "
            "Your goal is to answer the user's numeric question using ONLY this metadata. "
            "Perform the calculation internally and output ONLY the final number. "
            "Do not output units, equations, or sentences."
        )
        
        user_prompt = (
            f"Metadata Context:\n{context_str}\n\n"
            f"Question: '{query}'\n"
            "Answer:"
        )
        
        # 4. Ask Qwen (No Heuristics, No Python Eval)
        answer = self.ask_qwen(image, user_prompt, system_prompt=sys_prompt, max_tokens=32)
        
        # Clean up any lingering text just in case
        cleaned_answer = re.sub(r"[^\d\.]", "", answer)
        return cleaned_answer

    def solve_general_vqa(self, image, query, detections, type_q="binary", gsd=1.0):
        print(f"--- Task: {type_q.capitalize()} VQA ('{query}') ---")
        
        # 1. Ensure Data
        detections = self.ensure_detections_for_query(image, query, detections, gsd)
        
        # 2. Prepare Visuals (Annotate relevant objects)
        target_keywords = self.extract_target_classes(image, query)
        relevant_obbs = [d['obb'] for d in detections if d['label'] in target_keywords]
        
        if relevant_obbs:
            # If specific objects found, highlight them
            visual_input, _ = self.annotate_image_with_boxes(image, relevant_obbs)
            visual_note = "The image has been annotated with RED BOXES and IDs to help you locate the objects."
        else:
            # If nothing found, show everything to confirm absence or context
            all_obbs = [d['obb'] for d in detections]
            if all_obbs:
                visual_input, _ = self.annotate_image_with_boxes(image, all_obbs)
                visual_note = "The image is annotated with all detected objects."
            else:
                visual_input = image
                visual_note = "No specific objects were detected in the metadata."

        # 3. Define System Behavior
        if type_q == "binary":
            sys_prompt = "You are a strict answering machine. Answer the question with 'Yes' or 'No' ONLY."
        else:
            sys_prompt = "You are a concise assistant. Answer the question with a single word or short phrase if required"

        context_str = self.format_grounding_context(detections)
        
        user_prompt = (
            f"Metadata Context:\n{context_str}\n\n"
            f"Visual Context: {visual_note}\n"
            f"Question: '{query}'"
        )
        
        return self.ask_qwen(visual_input, user_prompt, system_prompt=sys_prompt, max_tokens=32)
        
'''
    def solve_numeric_vqa_run(self, image, query, detections,gsd):
        """
        Task: Numeric VQA
        Input: Original Image + RICH Text Context (Metadata)
        Output: Computed Answer
        """
        print(f"--- Task: Numeric VQA ('{query}') ---")
        detections = self.ensure_detections_for_query(image, query, detections, gsd=gsd)
        
        ans, success = self.solve_heuristic_vqa(query, detections)
        if success:
            print(f"   -> Solved Heuristically: {ans}")
            return ans
            
        context_str = self.format_grounding_context(detections)

        prompt = (
            f"Context:\n{context_str}\n\n"
            f"Question: '{query}'\n"
            "Task: Write a Python expression to calculate the answer.\n"
            "Rules:\n"
            "- Use list comprehensions or detected values.\n"
            "- Variable 'detections' is a list of dicts with keys: 'area_m2', 'center_point' (tuple), 'label'.\n"
            "- Example Area: 'sum([d['area_m2'] for d in detections])'\n"
            "- Example Dist: 'math.dist(detections[0]['center_point'], detections[1]['center_point']) * 1.57'\n"
            "Return ONLY the Python expression."
        )
        
        equation_response = self.ask_qwen(image, prompt, max_tokens=64)
        clean_eq = re.sub(r"[^a-zA-Z0-9_\[\]\(\)\+\-\*\/\.\,\'\<>=: ]", "", equation_response).strip()
        
        eval_context = {
            "detections": detections, 
            "math": math, 
            "sum": sum, "max": max, "min": min, "len": len
        }
        print(f"   -> Generated Logic: {clean_eq}")

        try:
            result = eval(clean_eq, {"__builtins__": {}}, eval_context)
            return str(result)
        except Exception as e:
            print(f"   -> Eval failed: {e}")
            return "Error calculating result"
            

    def solve_general_vqa(self, image, query, detections, type_q="binary", gsd = 1.0):
        """
        Task: Binary & Semantic VQA
        """
        print(f"--- Task: {type_q.capitalize()} VQA ('{query}') ---")
        detections = self.ensure_detections_for_query(image, query, detections, gsd=gsd)
        
        # 2. Try Heuristic First (Fast Path)
        ans, success = self.solve_heuristic_vqa(query, detections)
        if success:
            print(f"   -> Solved Heuristically: {ans}")
            return ans

        target_keywords = self.extract_target_classes(image, query)
        
        relevant_obbs = [d['obb'] for d in detections if d['label'] in target_keywords]
        
        # Logic: Draw ONLY relevant objects if found. 
        # Fallback: Draw ALL objects if no specific target matches (e.g., broad context question).
        if relevant_obbs:
            obbs_to_draw = relevant_obbs
        else:
            obbs_to_draw = [d['obb'] for d in detections] if detections else []

        # 4. Annotate Image
        if obbs_to_draw:
            visual_input, _ = self.annotate_image_with_boxes(image, obbs_to_draw)
            visual_instruction = (
                "I have annotated the image with RED BOUNDING BOXES and IDs. "
                "You MUST look inside these boxes to answer."
            )
        else:
            visual_input = image 
            visual_note = "No specific objects were detected, look at the whole image."

        # 5. Prepare Prompt
        plt.figure(figsize=(10, 10))
        plt.imshow(visual_input)
        plt.axis('off')
        plt.title(f"VQA Context: {query}")
        plt.show()
        
        context_str = self.format_grounding_context(detections)
        
        prompt = (
            f"Image Context:\n{visual_instruction}\n\n"
            f"Object Metadata (Location/Size):\n{context_str}\n\n"
            f"User Question: '{query}'\n\n"
            "Instructions:\n"
            "1. Use the Object IDs to locate the specific regions in the image.\n"
            "2. VISUALLY inspect the pixels inside the box to determine attributes (color, material, activity).\n"
            "3. Do NOT rely solely on the metadata, as it does not contain visual appearance info.\n"
            "Answer:"
        )
        
        answer = self.ask_qwen(visual_input, prompt, max_tokens=128)
        return answer

'''

In [ ]:
def main(json_path="sample2_query.json", output_path="final_output.json"):
    print(f"--- Starting Main Orchestrator ---")
    
    if not os.path.exists(json_path):
        print(f"Error: Input file '{json_path}' not found.")
        return

    with open(json_path, 'r') as f:
        data = json.load(f)
        
    # Load Image
    image_info = data.get("input_image", {})
    image_id = image_info.get("image_id", "image.png")
    image_url = image_info.get("image_url")
    
    if os.path.exists(image_id):
        print(f"Loading local image: {image_id}")
        image = Image.open(image_id).convert("RGB")
    elif image_url:
        print(f"Downloading image from {image_url}...")
        try:
            response = requests.get(image_url, timeout=10)
            image = Image.open(BytesIO(response.content)).convert("RGB")
            image.save(image_id) 
        except Exception as e:
            print(f"Failed to download image: {e}")
            return
    else:
        print("Error: No valid image found.")
        return

    # Metadata
    metadata = image_info.get("metadata", {})
    gsd = metadata.get("spatial_resolution_m", 1.0)
    print(f"Using GSD: {gsd} m/pixel")

    # Initialize Pipeline
    try:
        pipeline = RS_Pipeline() 
    except Exception as e:
        print(f"Pipeline Init Failed: {e}")
        return

    results = {
        "image_id": image_id,
        "caption": "",
        "grounding_results": [],
        "vqa_answers": {}
    }
    
    queries = data.get("queries", {})

    # A. Captioning
    if "caption_query" in queries:
        instruction = queries["caption_query"]["instruction"]
        results["caption"] = pipeline.process_captioning(image, instruction)
        print(f"Caption: {results['caption'][:100]}...")

    # B. Grounding
    detections = [] 
    if "grounding_query" in queries:
        instruction = queries["grounding_query"]["instruction"]
        detections = pipeline.process_grounding(image, instruction, gsd=gsd)
        
        serializable_dets = []
        for det in detections:
            serializable_dets.append({
                "id": det["id"],
                "label": det["label"],
                "area_m2": det["area_m2"],
                "box_2d": det["obb"][0]
            })
        results["grounding_results"] = serializable_dets

    # C. VQA
    if "attribute_query" in queries:
        attr_queries = queries["attribute_query"]
        vqa_results = {}
        
        for q_type, q_data in attr_queries.items():
            question = q_data["instruction"]
            
            if q_type == "numeric":
                # Route 1: Numeric (Logic/Math)
                answer = pipeline.solve_numeric_vqa_run(image, question, detections, gsd=gsd)
            elif q_type in ["binary", "semantic"]:
                # Route 2: Visual Reasoning
                answer = pipeline.solve_general_vqa(image, question, detections, type_q=q_type, gsd=gsd)
            else:
                # Fallback
                answer = pipeline.solve_general_vqa(image, question, detections, type_q="general", gsd=gsd)
            
            vqa_results[q_type] = {
                "question": question,
                "answer": answer
            }
            print(f"VQA ({q_type}): {answer}")
            
        results["vqa_answers"] = vqa_results

    with open(output_path, 'w') as f:
        json.dump(results, f, indent=4)
    
    print(f"--- Pipeline Finished. Results saved to {output_path} ---")

if not os.path.exists("sample2_query.json"):
    sample_data = {
        "input_image": {
            "image_id": "sample2.png",
            "image_url": "https://bit.ly/4oYfvr0",
            "metadata": {"width": 512, "height": 512, "spatial_resolution_m": 1.57}
        },
        "queries": {
            "caption_query": {"instruction": "Generate a detailed caption."},
            "grounding_query": {"instruction": "Locate the track field."},
            "attribute_query": {
                "binary": {"instruction": "Is there any aeroplane?"},
                "numeric": {"instruction": "What is the area of the track field?"},
                "semantic": {"instruction": "What is the color of the building?"}
            }
        }
    }
    with open("sample2_query.json", "w") as f:
        json.dump(sample_data, f, indent=4)
            
main()